# Téléchargement du réseau piéton - Canton de Genève

Ce notebook permet de télécharger et analyser les données de réseau piéton du Canton de Genève depuis OpenStreetMap, en utilisant les classes spécifiques à la marche (pedestrian, footway, steps, etc.).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('../../'))

# Imports
import geopandas as gpd
import osmnx as ox
import pandas as pd
import matplotlib.pyplot as plt
import folium
from shapely.geometry import Point, LineString
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print("📦 Libraries importées avec succès !")

In [ ]:
output_dir = "../../Data/input/network"

### 1. Définition de la zone d'étude

In [ ]:
%%time
# Récupérer les limites du Canton de Genève depuis OSM
print("🗺️ Téléchargement des limites du Canton de Genève...")

# Option 1: Par nom administratif (recommandé)
try:
    canton_geneve = ox.geocode_to_gdf("Canton de Genève, Switzerland")
    print("✅ Canton de Genève trouvé par nom administratif")
except:
    # Option 2: Fallback avec recherche plus générale
    try:
        canton_geneve = ox.geocode_to_gdf("Geneva Canton, Switzerland")
        print("✅ Canton de Genève trouvé avec nom anglais")
    except:
        # Option 3: Coordonnées manuelles si les autres échouent
        print("⚠️ Utilisation de coordonnées approximatives")
        # Bounding box approximatif du Canton de Genève
        north, south, east, west = 46.4, 46.1, 6.3, 5.9
        canton_geneve = gpd.GeoDataFrame(
            {'name': ['Geneva Canton']}, 
            geometry=[box(west, south, east, north)], 
            crs='EPSG:4326'
        )

print(f"📐 Superficie: {canton_geneve.to_crs('EPSG:2056').area.iloc[0]/1e6:.1f} km²")
canton_geneve.head()

In [ ]:
# # Import manquant pour la bounding box
# from shapely.geometry import box

# # Visualisation rapide de la zone
# m_canton = folium.Map(location=[46.25, 6.15], zoom_start=10, tiles='CartoDB Positron')

# # Ajouter les limites du canton
# folium.GeoJson(
#     canton_geneve.__geo_interface__,
#     style_function=lambda x: {
#         'fillColor': '#3489db',
#         'color': '#E60000',
#         'weight': 2,
#         'fillOpacity': 0.1,
#     }
# ).add_to(m_canton)

# print("🗺️ Carte des limites créée (exécuter m_canton pour l'afficher)")
# m_canton  # Décommenter pour afficher

### 2. Helpers

In [ ]:
import geopandas as gpd
import osmnx as ox

def get_platforms_from_polygon(poly, tags):
    """
    Version-agnostic OSMnx getter:
    - essaie ox.features_from_polygon (OSMnx ≥ 1.1)
    - sinon ox.geometries_from_polygon (OSMnx ~1.0)
    - sinon fallback bbox: ox.geometries_from_bbox + clip au polygone
    """
    # 1) versions récentes
    try:
        return ox.features_from_polygon(poly, tags=tags)
    except AttributeError:
        pass
    # 2) versions un peu plus anciennes
    try:
        return ox.geometries_from_polygon(poly, tags=tags)
    except AttributeError:
        pass
    # 3) fallback très compatible: BBOX + clip
    minx, miny, maxx, maxy = poly.bounds
    north, south, east, west = maxy, miny, maxx, minx
    gdf_bbox = ox.geometries_from_bbox(north, south, east, west, tags=tags)
    # Certaines vieilles versions n’ont pas gpd.clip; sinon on peut filtrer via intersection
    try:
        return gpd.clip(gdf_bbox, poly)
    except Exception:
        return gdf_bbox[gdf_bbox.geometry.intersects(poly)]
    

platform_tags = {
    "highway": "platform",
    "public_transport": "platform",
    "railway": "platform",
    "amenity": "bus_station",
}
plats = get_platforms_from_polygon(canton_geneve.geometry.iloc[0], tags=platform_tags)
print("Objets plateformes bruts:", 0 if plats is None else len(plats))

In [ ]:
# Export plats to geojson
plats.to_file(os.path.join(output_dir, "platforms.geojson"), driver='GeoJSON')

### 3. Téléchargement du réseau piéton

Nous allons télécharger spécifiquement les infrastructures dédiées à la marche selon les classifications OpenStreetMap.

In [ ]:
# Définition des filtres OSM pour les infrastructures piétonnes
pedestrian_filters = {
    # Chemins spécifiquement piétons
    'footway': True,                    # Trottoirs et chemins piétons
    'pedestrian': True,                 # Zones piétonnes
    'steps': True,                      # Escaliers
    'path': ['foot', 'designated'],     # Sentiers autorisés aux piétons
    
    # Routes avec accès piéton
    'highway': [
        'footway',
        'pedestrian', 
        'steps',
        'path',
        'living_street',                # Zones de rencontre
        'residential',                  # Rues résidentielles (avec trottoirs)
        'tertiary',                     # Routes tertiaires (souvent avec trottoirs)
        'unclassified',                 # Routes non classées
        'service',                      # Voies de service
        'platform'
    ],
    
    # Ajout pour public_transport
    'public_transport': [
        'platform'           
    ],
    
    # Attributs spéciaux pour les piétons
    'foot': ['yes', 'designated', 'permissive'],
    'sidewalk': ['both', 'left', 'right', 'yes'],  # Présence de trottoirs
    
    # Attributs spécifiques aux passages piétons
    'crossing': ['traffic_signals', 'uncontrolled', 'zebra', 'unmarked'],
    'crossing:signals': ['yes'],
    'tactile_paving': ['yes'],          # Bandes podotactiles
}

print("🚶 Filtres OSM définis pour les infrastructures piétonnes")
print(f"📝 Types d'infrastructures incluses:")
for key, values in pedestrian_filters.items():
    if isinstance(values, list):
        print(f"   • {key}: {', '.join(values)}")
    else:
        print(f"   • {key}: {values}")

In [ ]:
%%time
# Téléchargement du réseau piéton pour le Canton de Genève
print("🔄 Téléchargement du réseau piéton en cours...")
print("⚠️ Cette opération peut prendre plusieurs minutes...")

# Télécharger le réseau avec les filtres piétons
pedestrian_network = ox.graph_from_polygon(
    polygon = canton_geneve.geometry.iloc[0],         # Utiliser directement la géométrie du canton
    network_type='all',  # Réseau piéton
    custom_filter=None,   # On va filtrer après
    simplify = True,        # Ne garder que les nœuds essentiels
    retain_all=True       # Garder tous les segments
)

print(f"✅ Réseau téléchargé: {len(pedestrian_network.nodes)} nœuds, {len(pedestrian_network.edges)} arêtes")


In [ ]:
# Conversion en GeoDataFrames pour analyse et export
print("🔄 Conversion en GeoDataFrames...")

# Convertir les nœuds et arêtes en GeoDataFrames
nodes_gdf = ox.graph_to_gdfs(pedestrian_network, edges=False)
edges_gdf = ox.graph_to_gdfs(pedestrian_network, nodes=False)

print(f"📊 Nœuds: {len(nodes_gdf)} points")
print(f"📊 Arêtes: {len(edges_gdf)} segments")

# Afficher les colonnes disponibles pour les arêtes
print(f"\n📋 Colonnes des arêtes: {list(edges_gdf.columns)}")

# Statistiques sur les types d'infrastructures
if 'highway' in edges_gdf.columns:
    highway_counts = edges_gdf['highway'].value_counts()
    print(f"\n🏗️ Types d'infrastructures (highway):")
    for highway_type, count in highway_counts.head(10).items():
        print(f"   • {highway_type}: {count}")
        
# Vérifier la présence de trottoirs
if 'sidewalk' in edges_gdf.columns:
    sidewalk_counts = edges_gdf['sidewalk'].value_counts()
    print(f"\n🚶 Présence de trottoirs:")
    for sidewalk_type, count in sidewalk_counts.items():
        print(f"   • {sidewalk_type}: {count}")

In [ ]:
# Filtrer spécifiquement les infrastructures dédiées aux piétons
print("🎯 Filtrage des infrastructures spécifiquement piétonnes...")

# Types d'infrastructures prioritaires pour les piétons
pedestrian_priority = ['footway', 'pedestrian', 'steps', 'path', 'bus_stop', 'track']
pedestrian_secondary = ['living_street', 'residential', 'service']

# Créer des sous-ensembles avec .copy() pour éviter SettingWithCopyWarning
priority_pedestrian = edges_gdf[edges_gdf['highway'].isin(pedestrian_priority)].copy()
secondary_pedestrian = edges_gdf[edges_gdf['highway'].isin(pedestrian_secondary)].copy()

# Segments avec trottoirs explicites
if 'sidewalk' in edges_gdf.columns:
    sidewalk_segments = edges_gdf[edges_gdf['sidewalk'].notna() & 
                                  edges_gdf['sidewalk'].isin(['both', 'left', 'right', 'yes'])].copy()
else:
    sidewalk_segments = gpd.GeoDataFrame()

print(f"🚶 Infrastructures prioritaires piétons: {len(priority_pedestrian)}")
print(f"🏘️ Infrastructures secondaires (résidentielles): {len(secondary_pedestrian)}")
print(f"🛤️ Segments avec trottoirs explicites: {len(sidewalk_segments)}")


In [ ]:
# Ajouter cette cellule après le filtrage des arêtes
print("🔗 Isolation des nœuds spécifiques aux réseaux prioritaires et secondaires...")

# Récupérer les nœuds uniques des arêtes prioritaires
if len(priority_pedestrian) > 0:
    # Les arêtes ont des index multi-niveaux (u, v, key) où u et v sont les nœuds
    priority_node_ids = set()
    for edge_index in priority_pedestrian.index:
        priority_node_ids.add(edge_index[0])  # nœud de départ (u)
        priority_node_ids.add(edge_index[1])  # nœud d'arrivée (v)
    
    # Filtrer les nœuds correspondants
    priority_nodes = nodes_gdf[nodes_gdf.index.isin(priority_node_ids)].copy()
else:
    priority_nodes = gpd.GeoDataFrame()

# Récupérer les nœuds uniques des arêtes secondaires
if len(secondary_pedestrian) > 0:
    secondary_node_ids = set()
    for edge_index in secondary_pedestrian.index:
        secondary_node_ids.add(edge_index[0])  # nœud de départ (u)
        secondary_node_ids.add(edge_index[1])  # nœud d'arrivée (v)
    
    # Filtrer les nœuds correspondants
    secondary_nodes = nodes_gdf[nodes_gdf.index.isin(secondary_node_ids)].copy()
else:
    secondary_nodes = gpd.GeoDataFrame()

# Nœuds avec trottoirs explicites
if len(sidewalk_segments) > 0:
    sidewalk_node_ids = set()
    for edge_index in sidewalk_segments.index:
        sidewalk_node_ids.add(edge_index[0])
        sidewalk_node_ids.add(edge_index[1])
    
    sidewalk_nodes = nodes_gdf[nodes_gdf.index.isin(sidewalk_node_ids)].copy()
else:
    sidewalk_nodes = gpd.GeoDataFrame()

print(f"🚶 Nœuds du réseau prioritaire: {len(priority_nodes)}")
print(f"🏘️ Nœuds du réseau secondaire: {len(secondary_nodes)}")
print(f"🛤️ Nœuds avec trottoirs: {len(sidewalk_nodes)}")

# Analyser les intersections entre réseaux
if len(priority_nodes) > 0 and len(secondary_nodes) > 0:
    shared_nodes = set(priority_nodes.index).intersection(set(secondary_nodes.index))
    print(f"🔗 Nœuds partagés entre prioritaire/secondaire: {len(shared_nodes)}")

# Analyser le degré de connectivité des nœuds prioritaires
if len(priority_nodes) > 0:
    # Calculer le degré de chaque nœud (nombre de connexions)
    node_degrees = {}
    for edge_index in priority_pedestrian.index:
        u, v = edge_index[0], edge_index[1]
        node_degrees[u] = node_degrees.get(u, 0) + 1
        node_degrees[v] = node_degrees.get(v, 0) + 1
    
    # Ajouter les degrés aux nœuds prioritaires
    priority_nodes.loc[:, 'degree'] = [node_degrees.get(node_id, 0) for node_id in priority_nodes.index]
    
    print(f"\n📊 Statistiques de connectivité des nœuds prioritaires:")
    print(f"   • Degré moyen: {priority_nodes['degree'].mean():.1f}")
    print(f"   • Degré maximum: {priority_nodes['degree'].max()}")
    print(f"   • Nœuds de forte connectivité (≥4 connexions): {len(priority_nodes[priority_nodes['degree'] >= 4])}")

## 3. Visualisation du réseau piéton

In [ ]:
# Créer une carte interactive du réseau piéton
print("🗺️ Création de la carte interactive...")

# Carte centrée sur Genève
center_lat, center_lon = 46.2043907, 6.1431577
m_pedestrian = folium.Map(
    location=[center_lat, center_lon], 
    zoom_start=11, 
    tiles='CartoDB Positron'  # Utiliser un fond clair
)

# Fonction pour définir les couleurs selon le type d'infrastructure
def get_color(highway_type):
    color_map = {
        'footway': '#E60000',        # Rouge - trottoirs
        'pedestrian': '#F8D27D',     # Jaune - zones piétonnes  
        'steps': '#3489db',          # Bleu - escaliers
        'path': '#96C8A6',           # Vert - sentiers
        'living_street': '#DFAB9A',  # Rose - zones de rencontre
        'residential': '#cccccc'     # Gris - résidentiel
    }
    return color_map.get(highway_type, '#666666')

# Ajouter les infrastructures prioritaires
if len(priority_pedestrian) > 0:
    for _, row in priority_pedestrian.head(1000).iterrows():  # Limiter pour performance
        folium.PolyLine(
            locations=[[coord[1], coord[0]] for coord in row.geometry.coords],
            color=get_color(row['highway']),
            weight=2,
            opacity=0.8,
            popup=f"Type: {row['highway']}"
        ).add_to(m_pedestrian)

# Ajouter les nœuds prioritaires sur la carte
if len(priority_nodes) > 0:
    for _, node in priority_nodes.head(500).iterrows():  # Limiter pour performance
        folium.CircleMarker(
            location=[node.geometry.y, node.geometry.x],
            radius=0.3 if node.get('degree', 0) < 4 else 1,  # Taille selon connectivité
            color='#E60000',
            fillColor='#E60000',
            fillOpacity=0.7,
            popup=f"Nœud prioritaire<br>Connexions: {node.get('degree', 'N/A')}"
        ).add_to(m_pedestrian)

# Légende
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 200px; height: 120px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p><b>Réseau piéton</b></p>
<p><i class="fa fa-minus" style="color:#E60000"></i> Trottoirs (footway)</p>
<p><i class="fa fa-minus" style="color:#F8D27D"></i> Zones piétonnes</p>
<p><i class="fa fa-minus" style="color:#3489db"></i> Escaliers</p>
<p><i class="fa fa-minus" style="color:#96C8A6"></i> Sentiers</p>
</div>
'''
m_pedestrian.get_root().html.add_child(folium.Element(legend_html))

print("✅ Carte créée (exécuter m_pedestrian pour l'afficher)")
m_pedestrian  # Décommenter pour afficher

## 4. Export et sauvegarde des données

In [ ]:
def export_data(gdf, filename_base, formats=['geoparquet', 'pickle']):
    """Exporter un GeoDataFrame en plusieurs formats"""
    exported_files = []

    # Correction automatique de la colonne 'osmid' si elle existe
    if 'osmid' in gdf.columns:
        gdf = gdf.copy()  # éviter SettingWithCopyWarning
        gdf['osmid'] = gdf['osmid'].apply(
            lambda x: x if isinstance(x, list) else [x] if pd.notnull(x) else []
        )

    # Correction pour toutes les colonnes contenant des listes (sauf booléens purs)
    for col in gdf.columns:
        # Si la colonne contient des listes, mais pas des listes de booléens purs
        if gdf[col].apply(lambda x: isinstance(x, list)).any():
            # Si la colonne contient des listes de booléens, convertir en string
            if gdf[col].apply(lambda x: isinstance(x, list) and all(isinstance(i, bool) for i in x)).any():
                gdf[col] = gdf[col].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x) if isinstance(x, bool) else x)
            else:
                gdf[col] = gdf[col].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)

    for format_name in formats:
        if format_name in export_formats:
            ext = export_formats[format_name]
            if format_name == 'shapefile':
                # Créer un sous-dossier pour chaque shapefile
                shp_dir = os.path.join(output_dir, f"shp_{filename_base}")
                os.makedirs(shp_dir, exist_ok=True)
                filepath = os.path.join(shp_dir, f"{filename_base}{ext}")
            else:
                filepath = os.path.join(output_dir, f"{filename_base}{ext}")

            try:
                if format_name == 'geoparquet':
                    gdf.to_parquet(filepath)
                elif format_name == 'shapefile':
                    gdf.to_file(filepath)
                elif format_name == 'geojson':
                    gdf.to_file(filepath, driver='GeoJSON')
                elif format_name == 'pickle':
                    gdf.to_pickle(filepath)

                exported_files.append(filepath)
                print(f"✅ {format_name.upper()}: {filepath}")

            except Exception as e:
                print(f"❌ Erreur {format_name}: {e}")

    return exported_files

In [ ]:
%%time
# Export des différents jeux de données
print("💾 Export des données en cours...")

# 1. Réseau complet (nœuds et arêtes)
print("\n📊 Export du réseau complet:")
export_data(nodes_gdf, "geneva_pedestrian_nodes", ['geoparquet', 'shapefile', 'geojson'])
export_data(edges_gdf, "geneva_pedestrian_edges_all", ['geoparquet', 'shapefile', 'geojson'])

# 2. Infrastructures prioritaires piétonnes uniquement
print("\n🚶 Export des infrastructures prioritaires:")
export_data(priority_pedestrian, "geneva_pedestrian_priority", ['geoparquet', 'shapefile', 'geojson'])

# 3. Infrastructures secondaires (résidentielles)
print("\n🏘️ Export des infrastructures secondaires:")
export_data(secondary_pedestrian, "geneva_pedestrian_secondary", ['geoparquet', 'shapefile', 'geojson'])

# 4. Ajouter dans la cellule d'export
print("\n🔗 Export des nœuds spécifiques:")
if len(priority_nodes) > 0:
    export_data(priority_nodes, "geneva_pedestrian_priority_nodes", ['geoparquet', 'shapefile', 'geojson'])

if len(secondary_nodes) > 0:
    export_data(secondary_nodes, "geneva_pedestrian_secondary_nodes", ['geoparquet', 'shapefile', 'geojson'])

if len(sidewalk_nodes) > 0:
    export_data(sidewalk_nodes, "geneva_pedestrian_sidewalk_nodes", ['geoparquet', 'shapefile', 'geojson'])

# Mettre à jour les statistiques
summary_stats = {
    'date_extraction': pd.Timestamp.now().isoformat(),
    'total_nodes': len(nodes_gdf),
    'total_edges': len(edges_gdf),
    'priority_pedestrian_count': len(priority_pedestrian),
    'priority_pedestrian_km': total_priority_km,
    'secondary_pedestrian_count': len(secondary_pedestrian),
    'secondary_pedestrian_km': total_secondary_km,
    'priority_nodes_count': len(priority_nodes),
    'secondary_nodes_count': len(secondary_nodes),
    'sidewalk_nodes_count': len(sidewalk_nodes),
    'shared_priority_secondary_nodes': len(shared_nodes) if len(priority_nodes) > 0 and len(secondary_nodes) > 0 else 0,
    'data_source': 'OpenStreetMap via OSMnx',
    'area': 'Canton de Geneve'
}

# 5. Limites du canton
print("\n🗺️ Export des limites cantonales:")
export_data(canton_geneve, "geneva_canton_boundaries", ['geoparquet', 'shapefile', 'geojson'])

# 6. Statistiques de synthèse
import json
stats_file = os.path.join(output_dir, "extraction_summary.json")
with open(stats_file, 'w') as f:
    json.dump(summary_stats, f, indent=2)

print(f"\n📈 Statistiques sauvegardées: {stats_file}")
print("\n🎉 Export terminé !")
print(f"\n📁 Tous les fichiers sont dans: {output_dir}")
